# 10 - LangGraph y Flujos de Trabajo

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [Introducción a LangGraph](#intro)
2. [Estados y Grafos](#estados)
3. [Flujo RAG con LangGraph](#rag)
4. [Auto-corrección](#correccion)
5. [Checkpoints y Persistencia](#checkpoints)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Crear grafos de estados con LangGraph
- Implementar flujos condicionales
- Añadir auto-corrección a sistemas RAG
- Usar checkpoints para persistencia

<a name="intro"></a>
## 1. Introducción a LangGraph

**LangGraph** es una librería de LangChain para crear flujos de trabajo como grafos de estados.

### ¿Por qué LangGraph?

- **Control explícito**: Define exactamente el flujo
- **Condicionales**: Diferentes caminos según resultados
- **Ciclos**: Permite iteraciones y re-intentos
- **Estado**: Mantiene información entre nodos
- **Persistencia**: Checkpoints para recuperación

In [1]:
# Install
#!pip install -q langchain langchain-groq langgraph langchain-huggingface faiss-cpu

In [ ]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("GROQ API Key: ")

from langchain_groq import ChatGroq
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("Configurado ✓")

Configurado ✓


<a name="estados"></a>
## 2. Estados y Grafos

En LangGraph, definimos:
- **State**: Datos que fluyen por el grafo
- **Nodes**: Funciones que procesan el estado
- **Edges**: Conexiones entre nodos

In [3]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# Define state
class SimpleState(TypedDict):
    messages: List[str]
    current_step: str

# Define nodes
def step_one(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 1 completado"]
    return {"messages": messages, "current_step": "one"}

def step_two(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 2 completado"]
    return {"messages": messages, "current_step": "two"}

def step_three(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 3 completado"]
    return {"messages": messages, "current_step": "three"}

# Build graph
workflow = StateGraph(SimpleState)
workflow.add_node("step_one", step_one)
workflow.add_node("step_two", step_two)
workflow.add_node("step_three", step_three)

# Add edges
workflow.add_edge(START, "step_one")
workflow.add_edge("step_one", "step_two")
workflow.add_edge("step_two", "step_three")
workflow.add_edge("step_three", END)

# Compile
app = workflow.compile()
print("Grafo compilado ✓")

Grafo compilado ✓


In [4]:
# Run the graph
result = app.invoke({"messages": ["Inicio"], "current_step": ""})

print("Resultado:")
for msg in result["messages"]:
    print(f"  - {msg}")

Resultado:
  - Inicio
  - Paso 1 completado
  - Paso 2 completado
  - Paso 3 completado


### Grafos con condicionales

In [5]:
from typing import Literal

class ConditionalState(TypedDict):
    query: str
    query_type: str
    response: str

def classify_query(state: ConditionalState) -> ConditionalState:
    """Classify the query type."""
    query = state["query"].lower()
    if "precio" in query or "costo" in query:
        return {**state, "query_type": "pricing"}
    elif "horario" in query or "hora" in query:
        return {**state, "query_type": "schedule"}
    else:
        return {**state, "query_type": "general"}

def handle_pricing(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Los precios son: Básico 99€, Pro 299€, Enterprise consultar."}

def handle_schedule(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Horario: Lunes a Viernes, 9:00 a 18:00."}

def handle_general(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Para más información, contacta con soporte@empresa.com"}

def route_query(state: ConditionalState) -> Literal["pricing", "schedule", "general"]:
    return state["query_type"]

# Build conditional graph
cond_workflow = StateGraph(ConditionalState)
cond_workflow.add_node("classify", classify_query)
cond_workflow.add_node("pricing", handle_pricing)
cond_workflow.add_node("schedule", handle_schedule)
cond_workflow.add_node("general", handle_general)

cond_workflow.add_edge(START, "classify")
cond_workflow.add_conditional_edges(
    "classify",
    route_query,
    {"pricing": "pricing", "schedule": "schedule", "general": "general"}
)
cond_workflow.add_edge("pricing", END)
cond_workflow.add_edge("schedule", END)
cond_workflow.add_edge("general", END)

cond_app = cond_workflow.compile()
print("Grafo condicional compilado ✓")

Grafo condicional compilado ✓


In [6]:
# Test conditional routing
queries = [
    "¿Cuál es el precio del plan básico?",
    "¿Cuál es el horario de atención?",
    "¿Tienen servicio en México?"
]

for q in queries:
    result = cond_app.invoke({"query": q, "query_type": "", "response": ""})
    print(f"Q: {q}")
    print(f"A: {result['response']}\n")

Q: ¿Cuál es el precio del plan básico?
A: Los precios son: Básico 99€, Pro 299€, Enterprise consultar.

Q: ¿Cuál es el horario de atención?
A: Horario: Lunes a Viernes, 9:00 a 18:00.

Q: ¿Tienen servicio en México?
A: Para más información, contacta con soporte@empresa.com



<a name="rag"></a>
## 3. Flujo RAG con LangGraph

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

# Create vector store
docs = [
    Document(page_content="El IBI se paga anualmente basado en el valor catastral."),
    Document(page_content="El IVTM grava la titularidad de vehículos matriculados."),
    Document(page_content="El ICIO se liquida al finalizar construcciones u obras."),
    Document(page_content="Las bonificaciones pueden reducir hasta un 90% el impuesto."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store creado ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store creado ✓


In [8]:
from typing import List
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    messages: List[BaseMessage]
    context: str
    response: str

def retrieve_context(state: RAGState) -> RAGState:
    """Retrieve relevant documents."""
    query = state["messages"][-1].content
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate_response(state: RAGState) -> RAGState:
    """Generate response using LLM."""
    query = state["messages"][-1].content
    context = state["context"]
    
    prompt = f"""Responde basándote en el contexto.
    
Contexto: {context}

Pregunta: {query}

Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

# Build RAG graph
rag_workflow = StateGraph(RAGState)
rag_workflow.add_node("retrieve", retrieve_context)
rag_workflow.add_node("generate", generate_response)

rag_workflow.add_edge(START, "retrieve")
rag_workflow.add_edge("retrieve", "generate")
rag_workflow.add_edge("generate", END)

rag_app = rag_workflow.compile()
print("RAG graph compilado ✓")

RAG graph compilado ✓


In [9]:
# Test RAG
result = rag_app.invoke({
    "messages": [HumanMessage(content="¿Qué es el IBI?")],
    "context": "",
    "response": ""
})

print(f"Respuesta: {result['response']}")

Respuesta: El IBI (Impuesto de Bienes Inmuebles) es un impuesto que se paga anualmente y se basa en el valor catastral de un inmueble. Se aplica a los propietarios de bienes inmuebles, como casas, apartamentos, terrenos, etc. El objetivo del IBI es recaudar fondos para las administraciones locales, que luego se utilizan para financiar servicios y infraestructuras públicas en la comunidad.


<a name="correccion"></a>
## 4. Auto-corrección

Añadimos un paso de verificación y corrección.

In [10]:
class CorrectionState(TypedDict):
    query: str
    context: str
    response: str
    corrected_response: str
    needs_correction: bool

def retrieve(state: CorrectionState) -> CorrectionState:
    docs = retriever.invoke(state["query"])
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate(state: CorrectionState) -> CorrectionState:
    prompt = f"Contexto: {state['context']}\nPregunta: {state['query']}\nRespuesta:"
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def check_response(state: CorrectionState) -> CorrectionState:
    """Check if response needs correction."""
    check_prompt = f"""¿La siguiente respuesta está basada en el contexto?
    
Contexto: {state['context']}
Respuesta: {state['response']}

Responde solo 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}

def correct_response(state: CorrectionState) -> CorrectionState:
    """Correct the response."""
    correct_prompt = f"""Mejora esta respuesta basándote solo en el contexto.
    
Contexto: {state['context']}
Respuesta original: {state['response']}

Respuesta mejorada:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

def route_correction(state: CorrectionState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# Build correction graph
corr_workflow = StateGraph(CorrectionState)
corr_workflow.add_node("retrieve", retrieve)
corr_workflow.add_node("generate", generate)
corr_workflow.add_node("check", check_response)
corr_workflow.add_node("correct", correct_response)

corr_workflow.add_edge(START, "retrieve")
corr_workflow.add_edge("retrieve", "generate")
corr_workflow.add_edge("generate", "check")
corr_workflow.add_conditional_edges("check", route_correction, {"correct": "correct", "end": END})
corr_workflow.add_edge("correct", END)

corr_app = corr_workflow.compile()
print("Grafo con corrección compilado ✓")

Grafo con corrección compilado ✓


In [11]:
# Test
result = corr_app.invoke({
    "query": "¿Cuándo se paga el IVTM?",
    "context": "",
    "response": "",
    "corrected_response": "",
    "needs_correction": False
})

print(f"Respuesta original: {result['response']}")
print(f"Necesitó corrección: {result['needs_correction']}")
if result['corrected_response']:
    print(f"Respuesta corregida: {result['corrected_response']}")

Respuesta original: El IVTM se paga anualmente, al igual que el IBI, pero en este caso, se grava la titularidad de vehículos matriculados. Por lo tanto, la respuesta es: anualmente.
Necesitó corrección: False


<a name="checkpoints"></a>
## 5. Checkpoints y Persistencia

In [12]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpointer
memory = MemorySaver()

# Compile with checkpointer
rag_with_memory = rag_workflow.compile(checkpointer=memory)

# Run with thread_id for session tracking
config = {"configurable": {"thread_id": "session1"}}

result = rag_with_memory.invoke({
    "messages": [HumanMessage(content="¿Qué impuestos hay?")],
    "context": "",
    "response": ""
}, config=config)

print(f"Respuesta: {result['response']}")

Respuesta: Hay varios impuestos, pero en este contexto, se mencionan específicamente el impuesto que se puede reducir con bonificaciones (hasta un 90%) y el Impuesto sobre Bienes Inmuebles (IBI), que se paga anualmente basado en el valor catastral.


<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio: Crear un flujo con múltiples pasos

In [13]:
# Exercise: Create a workflow that:
# 1. Receives a question
# 2. Classifies the question type
# 3. Retrieves relevant info
# 4. Generates response
# 5. Checks quality
# 6. Corrects if needed

In [15]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document

class SupermarketState(TypedDict):
    question: str
    category: str
    context: str
    response: str
    corrected_response: str
    needs_correction: bool

# Supermarket knowledge base
supermarket_docs = [
    Document(page_content="Nuestro supermercado abre de lunes a sábado de 8:00 a 22:00 y domingos de 9:00 a 20:00."),
    Document(page_content="Las ofertas semanales incluyen 2x1 en fruta, 20% en conservas y descuentos en carne fresca."),
    Document(page_content="Estamos ubicados en Calle Mayor 123, junto al parque central."),
    Document(page_content="Aceptamos pago con tarjeta, efectivo y Bizum, y ofrecemos envíos a domicilio en la ciudad."),
    Document(page_content="Contamos con secciones de frutería, carnicería, panadería y productos ecológicos."),
    Document(page_content="El pack de leche cuesta 1,50€ y el paquete de arroz 1 kg cuesta 1,20€."),
]

embeddings_sm = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore_sm = FAISS.from_documents(supermarket_docs, embeddings_sm)
retriever_sm = vectorstore_sm.as_retriever(search_kwargs={"k": 3})

# Nodes

def classify_question(state: SupermarketState) -> SupermarketState:
    q = state["question"].lower()
    if any(word in q for word in ["precio", "coste", "oferta", "ofertas", "descuento"]):
        return {**state, "category": "pricing"}
    elif any(word in q for word in ["horario", "abierto", "apertura", "cierre"]):
        return {**state, "category": "hours"}
    elif any(word in q for word in ["ubicación", "dirección", "dónde", "donde"]):
        return {**state, "category": "location"}
    elif any(word in q for word in ["producto", "existencia", "stock"]):
        return {**state, "category": "product"}
    else:
        return {**state, "category": "general"}


def retrieve_supermarket_info(state: SupermarketState) -> SupermarketState:
    docs = retriever_sm.invoke(state["question"])
    context = "\n".join([doc.page_content for doc in docs])
    return {**state, "context": context}


def generate_supermarket_response(state: SupermarketState) -> SupermarketState:
    prompt = f"""Eres un asistente para un supermercado. Responde usando solo el contexto disponible.\n\nCategoría: {state['category']}\nContexto: {state['context']}\nPregunta: {state['question']}\nRespuesta:"""
    response = llm.invoke(prompt)
    return {**state, "response": response.content}


def check_supermarket_quality(state: SupermarketState) -> SupermarketState:
    prompt = f"""Evalúa si la respuesta usa únicamente el contexto y está bien formulada.\n\nPregunta: {state['question']}\nContexto: {state['context']}\nRespuesta: {state['response']}\n\nResponde solo SI o NO."""
    check = llm.invoke(prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}


def correct_supermarket_response(state: SupermarketState) -> SupermarketState:
    prompt = f"""Corrige la respuesta para que sea precisa y basada solo en el contexto.\n\nContexto: {state['context']}\nRespuesta original: {state['response']}\n\nRespuesta corregida:"""
    corrected = llm.invoke(prompt)
    return {**state, "corrected_response": corrected.content}


def route_supermarket_correction(state: SupermarketState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# Build graph
supermarket_workflow = StateGraph(SupermarketState)
supermarket_workflow.add_node("classify", classify_question)
supermarket_workflow.add_node("retrieve", retrieve_supermarket_info)
supermarket_workflow.add_node("generate", generate_supermarket_response)
supermarket_workflow.add_node("check", check_supermarket_quality)
supermarket_workflow.add_node("correct", correct_supermarket_response)

supermarket_workflow.add_edge(START, "classify")
supermarket_workflow.add_edge("classify", "retrieve")
supermarket_workflow.add_edge("retrieve", "generate")
supermarket_workflow.add_edge("generate", "check")
supermarket_workflow.add_conditional_edges("check", route_supermarket_correction, {"correct": "correct", "end": END})
supermarket_workflow.add_edge("correct", END)

supermarket_app = supermarket_workflow.compile()
print("Supermarket chatbot workflow compilado ✓")

# Test the workflow
for question in [
    "¿A qué hora abrís el domingo?",
    "¿Tenéis ofertas en fruta esta semana?",
    "¿Dónde está vuestra tienda?",
    "¿El pack de arroz está disponible?"
]:
    result = supermarket_app.invoke({
        "question": question,
        "category": "",
        "context": "",
        "response": "",
        "corrected_response": "",
        "needs_correction": False,
    })

    final_answer = result["corrected_response"] or result["response"]
    print(f"\nPregunta: {question}")
    print(f"Categoría: {result['category']}")
    print(f"Respuesta: {final_answer}")
    print(f"Necesita corrección: {result['needs_correction']}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Supermarket chatbot workflow compilado ✓

Pregunta: ¿A qué hora abrís el domingo?
Categoría: general
Respuesta: El domingo abrimos de 9:00 a 20:00.
Necesita corrección: False

Pregunta: ¿Tenéis ofertas en fruta esta semana?
Categoría: pricing
Respuesta: Sí, esta semana tenemos una oferta especial en fruta: 2x1. Compra una unidad y lleva otra igual de forma gratuita. ¡No te pierdas esta oportunidad de ahorrar en tus compras de fruta fresca!
Necesita corrección: False

Pregunta: ¿Dónde está vuestra tienda?
Categoría: location
Respuesta: Lo siento, pero no tengo información sobre la ubicación exacta de nuestra tienda. Sin embargo, puedo decirte que ofrecemos envíos a domicilio en la ciudad, lo que sugiere que estamos ubicados en algún lugar dentro de la ciudad. Si necesitas más información, te recomiendo que nos llames o visites nuestra página web para obtener detalles de contacto y ubicación.
Necesita corrección: False

Pregunta: ¿El pack de arroz está disponible?
Categoría: general
Resp

## Resumen

En este notebook hemos aprendido:

1. **LangGraph**: Crear flujos como grafos de estados
2. **Condicionales**: Routing basado en resultados
3. **RAG workflow**: Retrieve → Generate
4. **Auto-corrección**: Verificar y mejorar respuestas
5. **Checkpoints**: Persistencia de sesiones

En el siguiente notebook veremos **RAG Avanzado Agentic** con flujos completos.

---

## Referencias

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)

In [14]:
import session_info
session_info.show(html=False)

-----
ipykernel                   7.2.0
langchain_community         0.4.1
langchain_core              1.2.22
langchain_groq              1.1.2
langchain_huggingface       NA
langgraph                   NA
session_info                v1.0.1
-----
IPython             9.11.0
jupyter_client      8.8.0
jupyter_core        5.9.1
jupyterlab          4.5.6
notebook            7.5.5
-----
Python 3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:05:56) [MSC v.1942 64 bit (AMD64)]
Windows-11-10.0.26200-SP0
-----
Session information updated at 2026-06-17 19:32
